# Notebook 01: Data Loading and Structural Inspection

## Purpose
This notebook initializes the Malawi DHS 2024 child stunting project by loading the core DHS recode datasets required for analysis and confirming their structural integrity.

## Objectives
The notebook has five objectives:

1. Import the core Python libraries required for data handling and project organization.
2. Define project paths in a reproducible way.
3. Load the main DHS datasets that will support the child stunting modeling pipeline.
4. Inspect dataset dimensions, variable structure, and key linking identifiers.
5. Confirm the availability and format of the anthropometric variables required to construct the stunting outcome.

## Datasets loaded
The notebook focuses on the three main datasets required for the first stage of the project:

- **KR**: Children under five recode
- **IR**: Women's individual recode
- **HR**: Household recode

These datasets provide the child-level outcome, maternal characteristics, and household context needed for downstream feature engineering and modeling.

## Expected output
By the end of this notebook, the project should have:

- correctly loaded DHS datasets,
- verified key linking variables,
- confirmed the presence of the child anthropometric variables,
- and established a reproducible project environment for subsequent notebooks.

## Notes
This notebook does **not** perform data cleaning, feature engineering, target construction, or modeling. Those tasks are handled in later notebooks.

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

In [2]:
print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

print("Project root:", PROJECT_ROOT)
print("Raw data path:", DATA_RAW)
print("Interim data path:", DATA_INTERIM)
print("Processed data path:", DATA_PROCESSED)

Python executable: /Users/frack/miniforge3/envs/ocr/bin/python
Python version: 3.10.18
Project root: /Users/frack/Documents/PhD Data Science/malawi-dhs-2024-geoai
Raw data path: /Users/frack/Documents/PhD Data Science/malawi-dhs-2024-geoai/data/raw
Interim data path: /Users/frack/Documents/PhD Data Science/malawi-dhs-2024-geoai/data/interim
Processed data path: /Users/frack/Documents/PhD Data Science/malawi-dhs-2024-geoai/data/processed


In [3]:
DATA_INTERIM.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

## 3. Load core DHS datasets

This section loads the three main DHS recode files required for the first stage of the stunting project:

- **KR**: child-level records for children under five
- **IR**: women's individual recode
- **HR**: household recode

These three datasets provide the core structure for linking child outcomes, maternal characteristics, and household context.

In [4]:
kr_path = DATA_RAW / "MWKR81FL.dta"
ir_path = DATA_RAW / "MWIR81FL.dta"
hr_path = DATA_RAW / "MWHR81FL.dta"

df_kr = pd.read_stata(kr_path)
df_ir = pd.read_stata(ir_path)
df_hr = pd.read_stata(hr_path)

print("KR shape:", df_kr.shape)
print("IR shape:", df_ir.shape)
print("HR shape:", df_hr.shape)

KR shape: (12312, 1209)
IR shape: (21587, 5879)
HR shape: (23095, 3067)


## 4. Inspect dataset dimensions and variable structure

This section checks the size of each dataset and previews the first variables in order to confirm the general structure and naming conventions.

In [5]:
summary_df = pd.DataFrame({
    "dataset": ["KR", "IR", "HR"],
    "rows": [df_kr.shape[0], df_ir.shape[0], df_hr.shape[0]],
    "columns": [df_kr.shape[1], df_ir.shape[1], df_hr.shape[1]],
    "memory_mb": [
        round(df_kr.memory_usage(deep=True).sum() / 1024**2, 2),
        round(df_ir.memory_usage(deep=True).sum() / 1024**2, 2),
        round(df_hr.memory_usage(deep=True).sum() / 1024**2, 2),
    ]
})

display(summary_df)

,dataset,rows,columns,memory_mb
0,KR,12312,1209,20.47
1,IR,21587,5879,241.35
2,HR,23095,3067,178.95


In [6]:
print("KR first 25 columns:")
print(df_kr.columns.tolist()[:25])

print("\nIR first 25 columns:")
print(df_ir.columns.tolist()[:25])

print("\nHR first 25 columns:")
print(df_hr.columns.tolist()[:25])

KR first 25 columns:
['caseid', 'bidx', 'v000', 'v001', 'v002', 'v003', 'v004', 'v005', 'v006', 'v007', 'v008', 'v008a', 'v009', 'v010', 'v011', 'v012', 'v013', 'v014', 'v015', 'v016', 'v017', 'v018', 'v019', 'v019a', 'v020']

IR first 25 columns:
['caseid', 'v000', 'v001', 'v002', 'v003', 'v004', 'v005', 'v006', 'v007', 'v008', 'v008a', 'v009', 'v010', 'v011', 'v012', 'v013', 'v014', 'v015', 'v016', 'v017', 'v018', 'v019', 'v019a', 'v020', 'v021']

HR first 25 columns:
['hhid', 'hv000', 'hv001', 'hv002', 'hv003', 'hv004', 'hv005', 'hv006', 'hv007', 'hv008', 'hv008a', 'hv009', 'hv010', 'hv011', 'hv012', 'hv013', 'hv014', 'hv015', 'hv016', 'hv017', 'hv018', 'hv019', 'hv020', 'hv021', 'hv022']


In [7]:
print("KR key identifiers:")
display(df_kr[["v001", "v002", "b16"]].head())

print("IR key identifiers:")
display(df_ir[["v001", "v002", "v003"]].head())

print("HR key identifiers:")
display(df_hr[["hv001", "hv002"]].head())

KR key identifiers:


,v001,v002,b16
0,1,1,NaN
1,1,9,4.0
2,1,16,5.0
3,1,16,4.0
4,1,24,7.0


IR key identifiers:


,v001,v002,v003
0,1,1,1
1,1,9,1
2,1,16,2
3,1,24,2
4,1,39,2


HR key identifiers:


,hv001,hv002
0,1,1
1,1,9
2,1,16
3,1,24
4,1,32


## 6. Inspect anthropometric variables for target construction

This section checks the main child anthropometric variables required to define the stunting outcome.

The main variable of interest is:

- **HW70**: height-for-age z-score (HAZ)

Other anthropometric variables are also inspected for context.

In [8]:
anthro_vars = ["hw70", "hw71", "hw72"]

print("Available anthropometric variables:")
print([col for col in anthro_vars if col in df_kr.columns])

display(df_kr[anthro_vars].head())
display(df_kr[anthro_vars].describe(include="all"))

Available anthropometric variables:
['hw70', 'hw71', 'hw72']


,hw70,hw71,hw72
0,NaN,NaN,NaN
1,-84.0,-14.0,57.0
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,-252.0,-139.0,13.0


,hw70,hw71,hw72
count,5447,5451.0,5445
unique,705,606.0,637
top,flagged cases,-128.0,flagged cases
freq,32,34.0,36


## 7. Summary of Notebook 01

At this stage, the project has:

- initialized the project environment,
- loaded the main DHS datasets required for the child stunting workflow,
- confirmed the key identifiers needed for linking child, maternal, and household records,
- and verified the availability of the anthropometric variables required to construct the stunting target.

No data cleaning or target construction has been performed yet. Those steps will be handled in Notebook 02.